# Traffic Demand Prediction Final

In [1]:
import pandas as pd
from google.colab import files

print("Upload train.csv")
uploaded = files.upload()
train = pd.read_csv('train.csv')

print("\nUpload test.csv")
uploaded = files.upload()
test = pd.read_csv('test.csv')

print(f"\n✅ Train shape: {train.shape}")
print(f"✅ Test shape:  {test.shape}")
train.head()

Upload train.csv


Saving train.csv to train.csv

Upload test.csv


Saving test.csv to test.csv

✅ Train shape: (77299, 11)
✅ Test shape:  (41778, 10)


,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


In [8]:

# Traffic Demand Prediction - Single Notebook

!pip -q install lightgbm

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import lightgbm as lgb

TARGET="demand"
ID_COL="Index"

train=pd.read_csv("train.csv")
test=pd.read_csv("test.csv")

def create_features(df):
    df=df.copy()
    df["timestamp"]=df["timestamp"].astype(str)

    parts=df["timestamp"].str.split(":",expand=True)
    df["hour"]=pd.to_numeric(parts[0],errors="coerce").fillna(0).astype(int)
    df["minute"]=pd.to_numeric(parts[1],errors="coerce").fillna(0).astype(int)

    df["time_slot"]=df["hour"]*2+(df["minute"]>=30).astype(int)

    df["hour_sin"]=np.sin(2*np.pi*df["hour"]/24)
    df["hour_cos"]=np.cos(2*np.pi*df["hour"]/24)
    df["day_sin"]=np.sin(2*np.pi*df["day"]/7)
    df["day_cos"]=np.cos(2*np.pi*df["day"]/7)

    df["is_weekend"]=(df["day"]>=5).astype(int)

    return df

train=create_features(train)
test=create_features(test)

combined=pd.concat([train.drop(columns=[TARGET]),test],ignore_index=True)

for col in combined.columns:
    if combined[col].dtype=="object":
        combined[col]=combined[col].fillna("Unknown")
        le=LabelEncoder()
        combined[col]=le.fit_transform(combined[col].astype(str))

for col in combined.columns:
    if combined[col].dtype!="object":
        combined[col]=combined[col].fillna(combined[col].median())

if "timestamp" in combined.columns:
    combined=combined.drop(columns=["timestamp"])

train_len=len(train)
train_enc=combined.iloc[:train_len].copy()
test_enc=combined.iloc[train_len:].copy()

train_enc[TARGET]=train[TARGET].values

global_mean=train[TARGET].mean()

if "geohash" in train_enc.columns:
    mean_map=train_enc.groupby("geohash")[TARGET].mean()
    train_enc["geohash_mean"]=train_enc["geohash"].map(mean_map)
    test_enc["geohash_mean"]=test_enc["geohash"].map(mean_map)

train_enc=train_enc.fillna(global_mean)
test_enc=test_enc.fillna(global_mean)

FEATURES=[c for c in train_enc.columns if c not in [TARGET,ID_COL]]

X=train_enc[FEATURES]
y=np.log1p(train_enc[TARGET])

kf=KFold(n_splits=5,shuffle=True,random_state=42)

oof=np.zeros(len(X))
preds=np.zeros(len(test_enc))

for tr_idx,val_idx in kf.split(X):
    Xtr,Xval=X.iloc[tr_idx],X.iloc[val_idx]
    ytr,yval=y.iloc[tr_idx],y.iloc[val_idx]

    model=lgb.LGBMRegressor(
        objective="regression",
        n_estimators=5000,
        learning_rate=0.015,
        num_leaves=127,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.3,
        reg_lambda=0.3,
        random_state=42
    )

    model.fit(
        Xtr,ytr,
        eval_set=[(Xval,yval)],
        callbacks=[lgb.early_stopping(200,verbose=False)]
    )

    oof[val_idx]=model.predict(Xval)
    preds+=model.predict(test_enc[FEATURES])/5

print("Validation R2:",r2_score(np.expm1(y),np.expm1(oof)))

submission=pd.DataFrame({
    "Index":test[ID_COL],
    "demand":np.expm1(preds)
})

print("Submission shape:",submission.shape)

submission.to_csv("submission.csv",index=False)
submission.head()


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006421 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 908
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 16
[LightGBM] [Info] Start training from score 0.083012
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005632 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 908
[LightGBM] [Info] Number of data points in the train set: 61839, number of used features: 16
[LightGBM] [Info] 

,Index,demand
0,0,0.044254
1,1,0.026842
2,2,0.028913
3,3,0.033099
4,4,0.050372


In [5]:
submission.to_csv("submission.csv", index=False)

print("submission.csv saved successfully")

submission.csv saved successfully


In [6]:
from google.colab import files
files.download("submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>